# 05 -- Label Build

> **Data mode: SYNTHETIC FIXTURE DATA.** Every filing/price/universe/sector record in this notebook comes from `_fixtures.py`, hand-authored for demonstration only. **Nothing computed in this notebook is a genuine historical result, and no cell here may be cited as evidence of reproducing `report_current.html`.** See `docs/reproducibility_findings.md` for the actual reproducibility status.

Runs `build_production_labels`, which mirrors -- call for call -- the same two Stage 6 functions (`build_forward_return_outcome`, `assign_quarterly_labels`) the Stage 7 backtest runner uses internally, so a standalone label build here can never diverge from what a full backtest would compute.

In [ ]:
import atlas_quant
print("atlas_quant path:", atlas_quant.__file__)

import sys
sys.path.insert(0, ".")  # this notebook's own directory (for _fixtures.py) -- not a legacy path hack
import _fixtures


In [ ]:
from datetime import date, datetime
from atlas_quant.strategies.filing_momentum_ml.config import FeatureCacheIdentity, FilingMomentumMLConfig
from atlas_quant.strategies.filing_momentum_ml.production.feature_label_build import (
    build_production_features, build_production_labels,
)
from atlas_quant.strategies.filing_momentum_ml.sector_encoding import SectorEncoder

bundle = _fixtures.build_synthetic_bundle()
calendar = _fixtures.build_trading_calendar(bundle)
periods = _fixtures.build_periods()
config = FilingMomentumMLConfig()
targets = [(i, p.quarter_end) for p in periods for i in bundle.universe]
cache_identity = FeatureCacheIdentity(
    strategy_id=config.strategy_id, strategy_version="notebook", feature_schema_version="1",
    fcf_mode=config.fcf_mode, train_years=config.ml_train_years, min_train_quarters=config.min_train_quarters,
    model_config_identity=config.identity(), universe_id="synthetic-notebook-universe",
    data_cutoff=date(2023, 6, 1), created_at=datetime(2023, 6, 1),
)
feature_result = build_production_features(
    config=config, calendar=calendar, sector_encoder=SectorEncoder(), targets=targets,
    filings_by_instrument=bundle.filings_by_instrument, prices_by_instrument=bundle.prices_by_instrument,
    sector_by_instrument=bundle.sector_by_instrument,
    data_cutoff=max(p.evaluation_timestamp for p in periods), cache_identity=cache_identity, cache_root=None,
)
observations_by_quarter = {}
for obs in feature_result.feature_pipeline_result.observations:
    observations_by_quarter.setdefault(obs.quarter_end, []).append(obs)

label_result = build_production_labels(
    periods=periods, observations_by_quarter=observations_by_quarter,
    prices_by_instrument=bundle.prices_by_instrument, n_winners=config.n_winners,
)
for quarter_end, rows in sorted(label_result.labeled_by_quarter.items()):
    positive = sum(1 for r in rows if r.label == 1)
    print(quarter_end, f"{len(rows)} row(s), {positive} positive")
for w in label_result.warnings:
    print("warning:", w)


## Findings

- With only 2 synthetic instruments and `n_winners=10` (report §4.2's default), every quarter here is a "small quarter" -- no positive labels are assigned, exactly matching the documented, legacy-consistent small-quarter policy (never inflate a thin quarter into artificial positive labels).

## Limitations

- A real universe (S&P 500 + Nasdaq 100) would essentially never hit the small-quarter path; this synthetic demonstration exercises a code path a real run would rarely take.